# 离线引擎 API

SGLang 提供了无需 HTTP 服务器的直接推理引擎，特别适用于额外的 HTTP 服务器会增加不必要复杂性或开销的场景。以下是两个常见用例：

- 离线批量推理
- 在引擎之上构建自定义服务器

本文档重点介绍离线批量推理，演示四种不同的推理模式：

- 非流式同步生成
- 流式同步生成
- 非流式异步生成
- 流式异步生成

此外，您可以方便地在 SGLang 离线引擎之上构建自定义服务器。详细示例请参见 [custom_server](https://github.com/sgl-project/sglang/blob/main/examples/runtime/engine/custom_server.py)。



## Nest Asyncio
请注意，如果您想在 ipython 或其他嵌套循环代码中使用**离线引擎**，需要添加以下代码：
```python
import nest_asyncio

nest_asyncio.apply()

```

## 高级用法

引擎支持 [VLM 推理](https://github.com/sgl-project/sglang/blob/main/examples/runtime/engine/offline_batch_inference_vlm.py) 以及[提取隐藏状态](https://github.com/sgl-project/sglang/blob/main/examples/runtime/hidden_states)。

更多用例请参见[示例](https://github.com/sgl-project/sglang/tree/main/examples/runtime/engine)。

## 离线批量推理

SGLang 离线引擎支持具有高效调度的批量推理。

In [ ]:
# launch the offline engine
import asyncio

import sglang as sgl
import sglang.test.doc_patch
from sglang.utils import async_stream_and_merge, stream_and_merge

llm = sgl.Engine(model_path="qwen/qwen2.5-0.5b-instruct")

### 非流式同步生成

In [ ]:
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]

sampling_params = {"temperature": 0.8, "top_p": 0.95}

outputs = llm.generate(prompts, sampling_params)
for prompt, output in zip(prompts, outputs):
    print("===============================")
    print(f"Prompt: {prompt}\nGenerated text: {output['text']}")

### 流式同步生成

In [ ]:
prompts = [
    "Write a short, neutral self-introduction for a fictional character. Hello, my name is",
    "Provide a concise factual statement about France’s capital city. The capital of France is",
    "Explain possible future trends in artificial intelligence. The future of AI is",
]

sampling_params = {
    "temperature": 0.2,
    "top_p": 0.9,
}

print("\n=== Testing synchronous streaming generation with overlap removal ===\n")

for prompt in prompts:
    print(f"Prompt: {prompt}")
    merged_output = stream_and_merge(llm, prompt, sampling_params)
    print("Generated text:", merged_output)
    print()

### 非流式异步生成

In [ ]:
prompts = [
    "Write a short, neutral self-introduction for a fictional character. Hello, my name is",
    "Provide a concise factual statement about France’s capital city. The capital of France is",
    "Explain possible future trends in artificial intelligence. The future of AI is",
]

sampling_params = {"temperature": 0.8, "top_p": 0.95}

print("\n=== Testing asynchronous batch generation ===")


async def main():
    outputs = await llm.async_generate(prompts, sampling_params)

    for prompt, output in zip(prompts, outputs):
        print(f"\nPrompt: {prompt}")
        print(f"Generated text: {output['text']}")


asyncio.run(main())

### 流式异步生成

In [ ]:
prompts = [
    "Write a short, neutral self-introduction for a fictional character. Hello, my name is",
    "Provide a concise factual statement about France’s capital city. The capital of France is",
    "Explain possible future trends in artificial intelligence. The future of AI is",
]

sampling_params = {"temperature": 0.8, "top_p": 0.95}

print("\n=== Testing asynchronous streaming generation (no repeats) ===")


async def main():
    for prompt in prompts:
        print(f"\nPrompt: {prompt}")
        print("Generated text: ", end="", flush=True)

        # Replace direct calls to async_generate with our custom overlap-aware version
        async for cleaned_chunk in async_stream_and_merge(llm, prompt, sampling_params):
            print(cleaned_chunk, end="", flush=True)

        print()  # New line after each prompt


asyncio.run(main())

In [ ]:
llm.shutdown()